In [0]:
%sql
CREATE table ecommerce.gold.dim_date(
    date_id INT,
    date_values DATE,
    year INT,
    month INT,
    day INT,
    month_name STRING,
    day_name STRING,
    day_type STRING,
    quarter STRING
    
);
INSERT INTO ecommerce.gold.dim_date(date_values)
SELECT explode(sequence(to_date('2023-01-01'), to_date('2025-12-31'), INTERVAL 1 DAY));

In [0]:
dim_date = spark.sql("""
select int(date_format(date_values,'yyyyMMdd')) as date_id, date_values, year(date_values) as year, month(date_values) as month, day(date_values) as day, date_format(date_values, 'MMMM') as month_name, date_format(date_values, 'EEEE') as day_name, 
(case when date_format(date_values, 'EEEE') == 'Sunday' or date_format(date_values, 'EEEE') == 'Saturday' then 'Weekend' else 'Weekday' end) as day_type,
CONCAT('Q', quarter(date_values), '-', year(date_values))as quarter
from ecommerce.gold.dim_date
""")

In [0]:
dim_date.write.mode('overwrite').format('delta').saveAsTable('ecommerce.gold.dim_date')

In [0]:
display(spark.sql("select * from ecommerce.gold.dim_date"))

date_id,date_values,year,month,day,month_name,day_name,day_type,quarter
20230101,2023-01-01,2023,1,1,January,Sunday,Weekend,Q1-2023
20230102,2023-01-02,2023,1,2,January,Monday,Weekday,Q1-2023
20230103,2023-01-03,2023,1,3,January,Tuesday,Weekday,Q1-2023
20230104,2023-01-04,2023,1,4,January,Wednesday,Weekday,Q1-2023
20230105,2023-01-05,2023,1,5,January,Thursday,Weekday,Q1-2023
20230106,2023-01-06,2023,1,6,January,Friday,Weekday,Q1-2023
20230107,2023-01-07,2023,1,7,January,Saturday,Weekend,Q1-2023
20230108,2023-01-08,2023,1,8,January,Sunday,Weekend,Q1-2023
20230109,2023-01-09,2023,1,9,January,Monday,Weekday,Q1-2023
20230110,2023-01-10,2023,1,10,January,Tuesday,Weekday,Q1-2023


In [0]:
raw_files = 's3://orders-ecommerce-de/raw'
processed_path = 's3://orders-ecommerce-de/processed'